In [1]:
%pip install -r ../requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_validate

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
import mlflow


import sys
import os

sys.path.append(os.path.abspath(".."))

# Importar la función de construcción del pipeline desde src.preprocessing
from src.preprocessing import build_pipeline
# Importar funciones de entrenamiento desde src.trainer
from src.trainer import run_training


mlflow.set_experiment("house_prices_models")

C:\Users\karen\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<Experiment: artifact_location='file:c:/Users/karen/Documents/proyecto_estadistica_multivariada/notebooks/mlruns/3', creation_time=1776070515897, experiment_id='3', last_update_time=1776070515897, lifecycle_stage='active', name='house_prices_models', tags={}, trace_location=None, workspace='default'>

In [3]:
X_train = pd.read_csv("../data/clean/X_train.csv")
y_train = pd.read_csv("../data/clean/y_train.csv").values.ravel()

## Entrenamiento modelos
Nota: estas métricas están para la variable en log

In [5]:
from sklearn.linear_model import LinearRegression

result_lr = run_training(
    model_name="linear_regression",
    model=LinearRegression(),
    pipeline_builder=build_pipeline,
    X=X_train,
    y=y_train,
    param_grid=None
)
result_lr

2026/04/15 01:49:38 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


{'model_name': 'linear_regression',
 'metrics': {'rmse_mean': np.float64(0.1568804609400624),
  'mse_mean': np.float64(0.025661942637630015),
  'mae_mean': np.float64(0.09222111123691429),
  'r2_mean': np.float64(0.8357476751244615)},
 'params': {},
 'model_path': '../artifacts/models\\linear_regression.pkl',
 'metrics_path': '../artifacts/metrics\\linear_regression_metrics.csv'}

In [6]:
from sklearn.linear_model import Ridge

ridge_grid = {
    "model__alpha": [0.1, 1.0, 10.0, 100.0]
}

result_ridge = run_training(
    model_name="ridge",
    model=Ridge(),
    pipeline_builder=build_pipeline,
    X=X_train,
    y=y_train,
    param_grid=ridge_grid
)

result_ridge

Fitting 5 folds for each of 4 candidates, totalling 20 fits


2026/04/15 01:50:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


{'model_name': 'ridge',
 'metrics': {'rmse': np.float64(0.11307623411355457),
  'mse': 0.012786234721303401,
  'mae': 0.07744931850150925,
  'r2': 0.9161241770994373},
 'params': {'model__alpha': 10.0},
 'model_path': '../artifacts/models\\ridge.pkl',
 'metrics_path': '../artifacts/metrics\\ridge_metrics.csv'}

In [7]:
from sklearn.linear_model import Lasso


lasso_grid = {
    "model__alpha": [0.0001, 0.001, 0.01, 0.1, 1.0]
}

result_lasso = run_training(
    model_name="lasso",
    model=Lasso(max_iter=10000),
    pipeline_builder=build_pipeline,
    X=X_train,
    y=y_train,
    param_grid=lasso_grid,
    cv=5
)

result_lasso

Fitting 5 folds for each of 5 candidates, totalling 25 fits


2026/04/15 01:50:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


{'model_name': 'lasso',
 'metrics': {'rmse': np.float64(0.11908523677371102),
  'mse': 0.014181293617450816,
  'mae': 0.08099132124758361,
  'r2': 0.9069727955192008},
 'params': {'model__alpha': 0.001},
 'model_path': '../artifacts/models\\lasso.pkl',
 'metrics_path': '../artifacts/metrics\\lasso_metrics.csv'}

In [8]:
from sklearn.tree import DecisionTreeRegressor

cart_grid = {
    "model__max_depth": [None, 5, 10, 20],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4]
}

result_cart = run_training(
    model_name="decision_tree",
    model=DecisionTreeRegressor(random_state=42),
    pipeline_builder=build_pipeline,
    X=X_train,
    y=y_train,
    param_grid=cart_grid,
    cv=5
)

result_cart

Fitting 5 folds for each of 36 candidates, totalling 180 fits


2026/04/15 01:53:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


{'model_name': 'decision_tree',
 'metrics': {'rmse': np.float64(0.151095518229959),
  'mse': 0.022829855629179878,
  'mae': 0.1098551728638018,
  'r2': 0.8502394982310079},
 'params': {'model__max_depth': 5,
  'model__min_samples_leaf': 4,
  'model__min_samples_split': 2},
 'model_path': '../artifacts/models\\decision_tree.pkl',
 'metrics_path': '../artifacts/metrics\\decision_tree_metrics.csv'}

In [9]:
from sklearn.ensemble import RandomForestRegressor

rf_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 10, 20]
}

result_rf = run_training(
    model_name="random_forest",
    model=RandomForestRegressor(random_state=42),
    pipeline_builder=build_pipeline,
    X=X_train,
    y=y_train,
    param_grid=rf_grid
)

Fitting 5 folds for each of 6 candidates, totalling 30 fits


2026/04/15 01:53:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


## Comparar los modelos


In [10]:
X_test = pd.read_csv("../data/clean/X_test.csv")
y_test = pd.read_csv("../data/clean/y_test.csv").values.ravel()

In [14]:
def evaluate_saved_models(
    model_paths_dict,
    X_test,
    y_test,
    inverse_log=True
):
    
    import joblib
    import numpy as np
    import pandas as pd
    import os
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
    
    y_test = np.array(y_test).ravel()
    
    results = []
    
    for name, path in model_paths_dict.items():
        
        if not os.path.exists(path):
            print(f"Modelo no encontrado: {path}")
            continue
        
        try:
            model = joblib.load(path)
            y_pred = model.predict(X_test)
            
            if inverse_log:
                y_pred = np.expm1(y_pred)
                y_true = np.expm1(y_test)
            else:
                y_true = y_test
            
            rmse = np.sqrt(mean_squared_error(y_true, y_pred))
            mse = mean_squared_error(y_true, y_pred)
            mae = mean_absolute_error(y_true, y_pred)
            r2 = r2_score(y_true, y_pred)
            
            results.append({
                "model": name,
                "rmse_test": rmse,
                "mse_test": mse,
                "mae_test": mae,
                "r2_test": r2
            })
        
        except Exception as e:
            print(f"Error en modelo {name}: {e}")
            continue
    
    results_df = pd.DataFrame(results)
    
    if not results_df.empty:
        results_df = results_df.sort_values("rmse_test").reset_index(drop=True)
    
    return results_df

In [15]:
model_paths = {
    "lasso": result_lasso["model_path"],
    "decision_tree": result_cart["model_path"],
    "random_forest": result_rf["model_path"],
    "linear_regression": result_lr["model_path"],
    "ridge": result_ridge["model_path"]
    # "xgboost": result_xgb["model_path"],
}

In [16]:
results_test = evaluate_saved_models(
    model_paths,
    X_test,
    y_test
)

results_test

Error en modelo linear_regression: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.


C:\Users\karen\AppData\Roaming\Python\Python314\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\karen\AppData\Roaming\Python\Python314\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\karen\AppData\Roaming\Python\Python314\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\karen\AppData\Roaming\Python\Python314\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


,model,rmse_test,mse_test,mae_test,r2_test
0,ridge,25160.128667,6.330321e+08,16193.925688,0.917470
1,lasso,25890.657447,6.703261e+08,16210.296841,0.912608
2,random_forest,29882.286226,8.929510e+08,17275.994221,0.883584
3,decision_tree,40935.812965,1.675741e+09,23788.357804,0.781529


In [18]:
results_test.to_csv("../data/results/test_results.csv", index=False)

In [19]:
def select_best_model(results_df, metric="rmse_test"):
    """
    Selecciona el mejor modelo según métrica
    
    Parameters
    ----------
    results_df : pd.DataFrame
    metric : str
        Métrica a minimizar (rmse_test recomendado)
    
    Returns
    -------
    best_row : pd.Series
    best_model_name : str
    """
    
    if metric not in results_df.columns:
        raise ValueError(f"{metric} no está en el DataFrame")
    
    # ordenar por métrica (menor es mejor)
    best_row = results_df.sort_values(metric).iloc[0]
    
    best_model_name = best_row["model"]
    
    print("🏆 Mejor modelo:")
    print(f"Modelo: {best_model_name}")
    print(f"{metric}: {best_row[metric]:.4f}")
    
    return best_row, best_model_name

In [20]:
# evaluar todos los modelos
results_test = evaluate_saved_models(
    model_paths,
    X_test,
    y_test
)

# seleccionar mejor
best_row, best_model_name = select_best_model(results_test)

Error en modelo linear_regression: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.
🏆 Mejor modelo:
Modelo: ridge
rmse_test: 25160.1287


C:\Users\karen\AppData\Roaming\Python\Python314\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\karen\AppData\Roaming\Python\Python314\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\karen\AppData\Roaming\Python\Python314\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\karen\AppData\Roaming\Python\Python314\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


In [21]:
import joblib

best_model_path = model_paths[best_model_name]
best_model = joblib.load(best_model_path)

In [22]:
joblib.dump(best_model, "../artifacts/final_model.pkl")

['../artifacts/final_model.pkl']